# Task 2 — Incremental CPG Parser Service

This executed notebook demonstrates the Python `ast` parser, the versioned
Kafka event contract, AST/CFG/DFG/call edges, deterministic stable IDs, and
bounded one-file-at-a-time processing. Repository discovery evidence is kept
separately in `task1_discovery_evidence.ipynb`.


---
## 0. Setup — Import & Paths

In [1]:
import sys
import os
import json

# Thêm thư mục src vào Python path
REPO_ROOT     = os.path.abspath('../lerobot')
PROJECT_ROOT  = os.path.abspath('..')
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print(f'Project root : {PROJECT_ROOT}')
print(f'Lerobot repo : {REPO_ROOT}')
print(f'Python version: {sys.version}')

Project root : /Users/giabao/Documents/bigdata/lab4/bigdata-lab04-pipeline
Lerobot repo : /Users/giabao/Documents/bigdata/lab4/bigdata-lab04-pipeline/lerobot
Python version: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.4.2)]


In [2]:
# Load the deterministic manifest as Task 2 input without repeating Task 1 output.
from discovery import discover_python_files

py_files = discover_python_files(REPO_ROOT)
assert py_files, "Task 2 requires a non-empty Python source manifest"


---
## Kafka Topic Schema (Giao diện cho Thành viên 2)

Tất cả schema và tên topic được định nghĩa trong `src/schemas.py`.
Thành viên 2 chỉ cần import và dùng — **không cần tự định nghĩa**.

In [3]:
from schemas import (
    TOPIC_NODES, TOPIC_EDGES, TOPIC_METADATA, TOPIC_ERRORS,
    make_node_event, make_edge_event, make_metadata_event, make_error_event
)

print('=== Kafka Topic Layout ===')
print(f'  cpg.nodes      → {TOPIC_NODES}')
print(f'  cpg.edges      → {TOPIC_EDGES}')
print(f'  cpg.metadata   → {TOPIC_METADATA}')
print(f'  cpg.errors     → {TOPIC_ERRORS}')

print('\n=== Sample Node Event Schema ===')
sample_node = make_node_event(
    node_id='node_abc123def456',
    file_path='src/lerobot/__init__.py',
    label='FunctionDef',
    node_type='FunctionDef',
    line_number=10,
    col_offset=0,
    end_lineno=25,
    end_col_offset=4,
    name='my_function',
    code_snippet='def my_function(arg1, arg2):',
    scope='MyClass',
)
print(json.dumps(sample_node, indent=2))

=== Kafka Topic Layout ===
  cpg.nodes      → cpg.nodes
  cpg.edges      → cpg.edges
  cpg.metadata   → cpg.metadata
  cpg.errors     → cpg.errors

=== Sample Node Event Schema ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.001998Z",
  "topic": "cpg.nodes",
  "repo_id": "",
  "file_id": "",
  "file_path": "src/lerobot/__init__.py",
  "file_hash": "",
  "parse_status": "success",
  "node_id": "node_abc123def456",
  "label": "FunctionDef",
  "properties": {
    "type": "FunctionDef",
    "line_number": 10,
    "col_offset": 0,
    "end_lineno": 25,
    "end_col_offset": 4,
    "name": "my_function",
    "code_snippet": "def my_function(arg1, arg2):",
    "scope": "MyClass"
  }
}


In [4]:
print('=== Sample Edge Event Schema ===')
sample_edge = make_edge_event(
    edge_id='edge_789xyz',
    file_path='src/lerobot/__init__.py',
    edge_type='CFG_BRANCH_TRUE',
    source_id='node_if_condition',
    target_id='node_then_body',
    properties={'branch': 'true_branch'},
)
print(json.dumps(sample_edge, indent=2))

print('\n=== Sample Error Event Schema ===')
sample_err = make_error_event(
    file_path='src/lerobot/broken.py',
    error_type='SyntaxError',
    error_message='invalid syntax',
    line_number=42,
    col_offset=7,
)
print(json.dumps(sample_err, indent=2))

=== Sample Edge Event Schema ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.004636Z",
  "topic": "cpg.edges",
  "repo_id": "",
  "file_id": "",
  "file_path": "src/lerobot/__init__.py",
  "file_hash": "",
  "parse_status": "success",
  "edge_id": "edge_789xyz",
  "type": "CFG_BRANCH_TRUE",
  "source_id": "node_if_condition",
  "target_id": "node_then_body",
  "properties": {
    "branch": "true_branch"
  }
}

=== Sample Error Event Schema ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.004679Z",
  "topic": "cpg.errors",
  "repo_id": "",
  "file_id": "",
  "file_path": "src/lerobot/broken.py",
  "file_hash": "",
  "parse_status": "error",
  "error_type": "SyntaxError",
  "error_message": "invalid syntax",
  "line_number": 42,
  "col_offset": 7
}


---
## Task 2: Incremental CPG Parser Service

### Thư viện được chọn: `ast` (Python Standard Library)

| Tiêu chí | `ast` stdlib | `tree-sitter` | `Joern` |
|----------|:-----------:|:-------------:|:-------:|
| Không cần cài | ✅ | ❌ | ❌ |
| Python-native | ✅ | Partial | ❌ |
| AST nodes | ✅ | ✅ | ✅ |
| CFG edges | ⚠️ Manual | ✅ | ✅ |
| DFG edges | ⚠️ Manual | ⚠️ Partial | ✅ |
| Call edges | ✅ Manual | ✅ | ✅ |

**Lý do chọn `ast`:** Phù hợp nhất cho môi trường học tập, không cần dependency ngoài, đủ để demonstrate 4 loại edge theo yêu cầu.

### Stable ID Strategy (quan trọng cho Task 6)

```
file_id = SHA-256(repo_id + normalized_relative_path)
node_id = SHA-256(file_id + structural_AST_path + node_type)
edge_id = SHA-256(file_id + edge_type + source_id + target_id + occurrence)
```

**Tại sao KHÔNG dùng `id(ast_node)`:**
- `id()` là địa chỉ bộ nhớ → thay đổi mỗi lần parse → duplicate nodes trong Neo4j
- Full SHA-256 của repo, file và structural AST path → deterministic, không phụ thuộc hệ điều hành

In [5]:
from parser_service import CPGParser

# Chọn một file production cố định để output có thể tái lập.
demo_info = next(f for f in py_files if f['relative_path'] == 'src/lerobot/utils/utils.py')
demo_file = demo_info['absolute_path']
print(f'Demo file: {demo_info["relative_path"]}')
print(f'Size: {demo_info["file_size_bytes"]} bytes')

Demo file: src/lerobot/utils/utils.py
Size: 12632 bytes


In [6]:
# Run parser
parser = CPGParser(absolute_path=demo_file, repo_root=REPO_ROOT)
nodes, edges, metadata, error_event = parser.parse()

if error_event:
    print(f'Parse error: {error_event["error_type"]} — {error_event["error_message"]}')
else:
    print('Parse thành công!')
    print(f'\nKết quả parse:')
    print(f'  Tổng AST nodes : {metadata["total_nodes"]}')
    print(f'  AST edges      : {metadata["total_edges"]["ast"]}')
    print(f'  CFG edges      : {metadata["total_edges"]["cfg"]}')
    print(f'  DFG edges      : {metadata["total_edges"]["dfg"]}')
    print(f'  CALL edges     : {metadata["total_edges"]["call"]}')
    print(f'  File hash      : {metadata["file_hash"]}')
    print(f'  Parse time     : {metadata["parse_duration_ms"]} ms')

Parse thành công!

Kết quả parse:
  Tổng AST nodes : 1633
  AST edges      : 1632
  CFG edges      : 166
  DFG edges      : 100
  CALL edges     : 86
  File hash      : 1052b37570376aadb56b1b69688c5b755440b7a5b081e762e966bda9fd75bae1
  Parse time     : 15.55 ms


In [7]:
# Xem chi tiết các loại node
import collections

node_types = collections.Counter(n['properties']['type'] for n in nodes)
print('=== Phân bổ loại AST Node (Top 15) ===')
for ntype, cnt in node_types.most_common(15):
    bar = '█' * cnt
    print(f'  {ntype:<25} {cnt:>4}  {bar}')

=== Phân bổ loại AST Node (Top 15) ===
  Load                       421  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Name                       337  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Constant                   133  ███████████████████████████████████████████████████████████████████████████████████████████████████

In [8]:
# Xem sample node event
print('=== Sample Node Event (sẽ produce lên cpg.nodes) ===')
# Tìm FunctionDef node để demo
func_nodes = [n for n in nodes if n['properties']['type'] == 'FunctionDef']
if func_nodes:
    print(json.dumps(func_nodes[0], indent=2))
else:
    print(json.dumps(nodes[0], indent=2))

=== Sample Node Event (sẽ produce lên cpg.nodes) ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.015394Z",
  "topic": "cpg.nodes",
  "repo_id": "lerobot",
  "file_id": "file_0e1f09520b7e4b98544d56df2d7dabfe3ae69514a437fffc788239afafe546c4",
  "file_path": "src/lerobot/utils/utils.py",
  "file_hash": "1052b37570376aadb56b1b69688c5b755440b7a5b081e762e966bda9fd75bae1",
  "parse_status": "success",
  "node_id": "node_3cfb6f33860bb4b81560e36aa6d74af218e90ba93d957b74f86f31715f26a889",
  "label": "FunctionDef",
  "properties": {
    "type": "FunctionDef",
    "line_number": 38,
    "col_offset": 0,
    "end_lineno": 41,
    "end_col_offset": 39,
    "name": "inside_slurm",
    "code_snippet": "def inside_slurm():",
    "scope": null
  }
}


In [9]:
# Xem phân bổ loại edge
edge_types = collections.Counter(e['type'] for e in edges)
print('=== Phân bổ loại Edge ===')
for etype, cnt in edge_types.most_common():
    bar = '█' * (cnt // 2 + 1)
    print(f'  {etype:<22} {cnt:>4}  {bar}')

=== Phân bổ loại Edge ===
  AST_CHILD              1632  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  CFG_NEXT                125  ███████████████████████████████████████████████████████████████
  DFG_USE                 100 

In [10]:
# Sample CFG edge
cfg_edges = [e for e in edges if e['type'].startswith('CFG')]
if cfg_edges:
    print('=== Sample CFG Edge ===')
    print(json.dumps(cfg_edges[0], indent=2))
else:
    print('Không có CFG edges trong file này (file không có control flow)')

=== Sample CFG Edge ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.022121Z",
  "topic": "cpg.edges",
  "repo_id": "lerobot",
  "file_id": "file_0e1f09520b7e4b98544d56df2d7dabfe3ae69514a437fffc788239afafe546c4",
  "file_path": "src/lerobot/utils/utils.py",
  "file_hash": "1052b37570376aadb56b1b69688c5b755440b7a5b081e762e966bda9fd75bae1",
  "parse_status": "success",
  "edge_id": "edge_ca43aedffc9c2601cc3be1e51a836c592b47395d62089f2095f333fc8e5c2f03",
  "type": "CFG_NEXT",
  "source_id": "node_1d0fb72367e84705951dcd50478b0062e0e176b19666438e44614a9c0838699d",
  "target_id": "node_98d67f55330bf9e570f3bc5f5dcb00108eb18e4794e40a96dd1e1ec925e6c9d6",
  "properties": {
    "sequence": 0
  }
}


In [11]:
# Sample DFG edge
dfg_edges = [e for e in edges if e['type'] == 'DFG_USE']
if dfg_edges:
    print('=== Sample DFG Edge ===')
    print(json.dumps(dfg_edges[0], indent=2))
    print(f'Biến được track: {dfg_edges[0]["properties"]["variable_name"]}')
else:
    print('Không có DFG edges trong file này')

=== Sample DFG Edge ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.024094Z",
  "topic": "cpg.edges",
  "repo_id": "lerobot",
  "file_id": "file_0e1f09520b7e4b98544d56df2d7dabfe3ae69514a437fffc788239afafe546c4",
  "file_path": "src/lerobot/utils/utils.py",
  "file_hash": "1052b37570376aadb56b1b69688c5b755440b7a5b081e762e966bda9fd75bae1",
  "parse_status": "success",
  "edge_id": "edge_892c03d38c2b85864b2ff51d6fc75ad9ac3b8c072024548ccb18940bef81b64f",
  "type": "DFG_USE",
  "source_id": "node_60b3362934b2977c2046d2088fa4c3fb64a68a80fec70b07a8fdf9df42c5e535",
  "target_id": "node_0f0a428e385ebdca23353237212fd5d171f8582e4ddb68d1fad2ffcbc4fe01bf",
  "properties": {
    "variable_name": "formatter"
  }
}
Biến được track: formatter


In [12]:
# Sample CALL edge
call_edges = [e for e in edges if 'CALL' in e['type']]
if call_edges:
    print('=== Sample CALL Edge ===')
    print(json.dumps(call_edges[0], indent=2))
else:
    print('Không có CALL edges trong file này')

=== Sample CALL Edge ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.025526Z",
  "topic": "cpg.edges",
  "repo_id": "lerobot",
  "file_id": "file_0e1f09520b7e4b98544d56df2d7dabfe3ae69514a437fffc788239afafe546c4",
  "file_path": "src/lerobot/utils/utils.py",
  "file_hash": "1052b37570376aadb56b1b69688c5b755440b7a5b081e762e966bda9fd75bae1",
  "parse_status": "success",
  "edge_id": "edge_3b161004383d7b762d05b8c7cc92e804bd59b8620a24163aa1e3ffc89749740e",
  "type": "CALL_EXTERNAL",
  "source_id": "node_99be69a185cb8deb3883696004a69f97bf03e0e2f079ba1334ec0fe236f79186",
  "target_id": "external::Formatter",
  "properties": {
    "callee_name": "Formatter",
    "call_site_id": "node_e3aa0a439a1aa92c1eb836eee6241c48f4954d757c79f9018d7e42a9f32f09aa",
    "call_site_line": 70,
    "is_external": true
  }
}


In [13]:
# Metadata event (sẽ produce lên cpg.metadata → Spark → MongoDB)
print('=== Metadata Event (cho cpg.metadata topic) ===')
print(json.dumps(metadata, indent=2))

=== Metadata Event (cho cpg.metadata topic) ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-25T14:50:23.026209Z",
  "topic": "cpg.metadata",
  "repo_id": "lerobot",
  "file_id": "file_0e1f09520b7e4b98544d56df2d7dabfe3ae69514a437fffc788239afafe546c4",
  "file_path": "src/lerobot/utils/utils.py",
  "file_size_bytes": 12632,
  "file_hash": "1052b37570376aadb56b1b69688c5b755440b7a5b081e762e966bda9fd75bae1",
  "parse_status": "success",
  "total_nodes": 1633,
  "total_edges": {
    "ast": 1632,
    "cfg": 166,
    "dfg": 100,
    "call": 86
  },
  "parser_version": "ast-stdlib-3.x",
  "parse_duration_ms": 15.55
}


---
## Kiểm chứng Stable ID & Idempotency (chuẩn bị cho Task 6)

**Yêu cầu:** Parse cùng 1 file nhiều lần phải sinh ra **cùng bộ node_id và edge_id**.
Điều này đảm bảo Neo4j dùng `MERGE` sẽ không tạo duplicate nodes/edges.

In [14]:
print('=== Kiểm tra Idempotency: parse 3 lần cùng file ===')

all_node_ids = []
all_edge_ids = []

for i in range(3):
    p = CPGParser(absolute_path=demo_file, repo_root=REPO_ROOT)
    n, e, m, err = p.parse()
    node_ids = sorted([x['node_id'] for x in n])
    edge_ids = sorted([x['edge_id'] for x in e])
    all_node_ids.append(node_ids)
    all_edge_ids.append(edge_ids)
    print(f'  Run {i+1}: {len(n)} nodes, {len(e)} edges')

nodes_stable = all(all_node_ids[0] == r for r in all_node_ids)
edges_stable = all(all_edge_ids[0] == r for r in all_edge_ids)

print()
print(f'Node IDs stable across 3 runs: {"YES" if nodes_stable else "NO"}')
print(f'Edge IDs stable across 3 runs: {"YES" if edges_stable else "NO"}')
print()
if nodes_stable and edges_stable:
    print('IDEMPOTENCY: PASS — Neo4j MERGE sẽ không tạo duplicate!')
else:
    print('IDEMPOTENCY: FAIL')

=== Kiểm tra Idempotency: parse 3 lần cùng file ===
  Run 1: 1633 nodes, 1984 edges


  Run 2: 1633 nodes, 1984 edges
  Run 3: 1633 nodes, 1984 edges

Node IDs stable across 3 runs: YES
Edge IDs stable across 3 runs: YES

IDEMPOTENCY: PASS — Neo4j MERGE sẽ không tạo duplicate!


In [15]:
# Demo: xem ID thực tế bằng đúng helper của Parser Service

print('=== Ví dụ về Stable ID vs. Memory-address ID ===')
import ast
from parser_service import _stable_file_id, _stable_node_id

test_code = '''
def greet(name):
    message = f"Hello, {name}!"
    return message
'''

ids_by_memory = []
ids_by_stable = []
demo_repo_id = 'huggingface/lerobot'
demo_file_id = _stable_file_id(demo_repo_id, 'test.py')

for _ in range(3):
    tree = ast.parse(test_code)
    func_node = next(n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef))
    
    # Cách cũ (BUG): dùng id() — memory address
    ids_by_memory.append(id(func_node))
    
    # Cách mới: full SHA-256 của file ID + structural path + node type
    ids_by_stable.append(_stable_node_id(
        'test.py', func_node, 'root.body[0]',
        repo_id=demo_repo_id, file_id=demo_file_id,
    ))

print(f'Memory-based IDs (BUG):  {ids_by_memory}')
print(f'All different?           {len(set(ids_by_memory)) > 1}')
print()
print(f'SHA256-based IDs (FIXED): {ids_by_stable}')
print(f'All same?                 {len(set(ids_by_stable)) == 1}')

=== Ví dụ về Stable ID vs. Memory-address ID ===
Memory-based IDs (BUG):  [4391520432, 4508863168, 4391524080]
All different?           True

SHA256-based IDs (FIXED): ['node_de6b81e5328a42ad30e335fb2642a6da59a7b628e259fcbba25145a6ac09fa6c', 'node_de6b81e5328a42ad30e335fb2642a6da59a7b628e259fcbba25145a6ac09fa6c', 'node_de6b81e5328a42ad30e335fb2642a6da59a7b628e259fcbba25145a6ac09fa6c']
All same?                 True


---
## Pipeline Demo: Xử lý 5 File Đầu Tiên

Demo toàn bộ pipeline incremental: mỗi file được parse độc lập, kết quả được in ra như thể đang produce lên Kafka.

In [16]:
print('=== Demo: Xử lý 5 file đầu tiên theo kiểu incremental ===')
print(f'{"File":<55} {"Nodes":>6} {"AST":>5} {"CFG":>5} {"DFG":>5} {"CALL":>5} {"ms":>6}')
print('-' * 95)

# Lấy file có content phong phú (size > 1KB)
interesting_files = [f for f in py_files if f['file_size_bytes'] > 1000][:5]

summary = []
for file_info in interesting_files:
    parser = CPGParser(absolute_path=file_info['absolute_path'], repo_root=REPO_ROOT)
    nodes, edges, meta, err = parser.parse()
    
    if err:
        print(f'{file_info["relative_path"]:<55} ERROR: {err["error_type"]}')
        continue
    
    e = meta['total_edges']
    print(f'{file_info["relative_path"]:<55} {meta["total_nodes"]:>6} {e["ast"]:>5} {e["cfg"]:>5} {e["dfg"]:>5} {e["call"]:>5} {meta["parse_duration_ms"]:>6.1f}')
    summary.append(meta)

print('-' * 95)
if summary:
    total_nodes = sum(m['total_nodes'] for m in summary)
    total_edges = sum(sum(m['total_edges'].values()) for m in summary)
    print(f'{"TOTAL (5 files)":<55} {total_nodes:>6} {total_edges:>22}')

=== Demo: Xử lý 5 file đầu tiên theo kiểu incremental ===
File                                                     Nodes   AST   CFG   DFG  CALL     ms
-----------------------------------------------------------------------------------------------
scripts/ci/extract_task_descriptions.py                   1003  1002    81   124    65   10.1
scripts/ci/parse_eval_metrics.py                           607   606    56    51    38    5.8
src/lerobot/__init__.py                                     81    80     6     1     1    0.8
src/lerobot/annotations/steerable_pipeline/__init__.py      19    18     4     0     0    0.2
src/lerobot/annotations/steerable_pipeline/config.py       624   623    87     2     8    5.8
-----------------------------------------------------------------------------------------------
TOTAL (5 files)                                           2334                   2853


---
## Task 2 summary

| Requirement | Evidence |
|---|---|
| Parser Service | Python standard-library `ast`, one file per invocation |
| AST edges | `AST_CHILD` events |
| CFG edges | sequential, branch and exception-flow approximation |
| DFG edges | definition-to-use analysis |
| Call edges | internal and external call events |
| Stable identity | deterministic full SHA-256 IDs, three-run replay `PASS` |
| Kafka handoff | four versioned event schemas with stable keys |

### Event handoff to Kafka integration

```python
from schemas import TOPIC_NODES, TOPIC_EDGES, TOPIC_METADATA, TOPIC_ERRORS
from parser_service import CPGParser

for file_info in py_files:
    parser = CPGParser(file_info['absolute_path'], REPO_ROOT)
    nodes, edges, metadata, error = parser.parse()

    for event in nodes:
        producer.send(TOPIC_NODES, value=event)
    for event in edges:
        producer.send(TOPIC_EDGES, value=event)
    producer.send(TOPIC_METADATA, value=metadata)
    if error:
        producer.send(TOPIC_ERRORS, value=error)
```
